In [2]:
import pandas as pd
import numpy as np
import time
import itertools

# ===================== SETTINGS =====================
DISPATCH_MODE = 'ceed'      # 'economic' | 'emission' | 'ceed'
PPF_H = None                # None -> auto compute global price-penalty; or set a float, e.g., 1.0
USE_VPE_IF_AVAILABLE = False  # True to include Valve-Point Effect if 'd' and 'e' columns exist

# ===================== LOAD DATA =====================
data = pd.read_excel(
    '/Users/workk/Documents/[Think!]/[FMK]/Research Things/Books/Optimasi/Dataset/13-unit.xlsx',
    sheet_name='Table 3'
)

def get_col(df, names, default=0.0):
    for n in names:
        if n in df.columns: 
            return df[n].values.astype(float)
    return np.full(len(df), default, dtype=float)

a = get_col(data, ['a'])
b = get_col(data, ['b'])
c = get_col(data, ['c'])
Minimum_Capacity = get_col(data, ['pmin','Pmin','Pmin (MW)'])
Maximum_Capacity = get_col(data, ['pmax','Pmax','Pmax (MW)'])
ramp_up        = get_col(data, ['rampup','RU','RampUp'])
ramp_down      = get_col(data, ['rampdown','RD','RampDown'])
Unit = get_col(data, ['Unit','unit','Unit ke-']).astype(int)

# Optional: VPE coefficients
d_vpe = get_col(data, ['d','D'], default=0.0)
e_vpe = get_col(data, ['e','E'], default=0.0)
use_vpe = USE_VPE_IF_AVAILABLE and (('d' in data.columns or 'D' in data.columns) and ('e' in data.columns or 'E' in data.columns))

# Emission coefficients (handle latin or Greek headers)
alpha = get_col(data, ['alpha','Alpha','α'])
beta  = get_col(data, ['beta','Beta','β'])
gamma = get_col(data, ['gamma','Gamma','γ'])
xi    = get_col(data, ['xi','Xi','ξ'])
delta = get_col(data, ['delta','Delta','δ'])

# Demand
power_demand_data = pd.read_excel('/Users/workk/Documents/[Think!]/[FMK]/Research Things/Books/Optimasi/Dataset/pd13u.xlsx')
power_demand = power_demand_data['load'].values.astype(float)

# ===================== COST & EMISSION =====================
def vpe_term(P, pmin, e, d):
    # | e_i * sin( d_i * (pmin_i - P_i) ) |
    return np.abs(e * np.sin(d * (pmin - P)))

def unit_cost(P, a, b, c, pmin, e=None, d=None, use_vpe=False):
    base = a + b*P + c*(P**2)
    if use_vpe and e is not None and d is not None:
        return base + vpe_term(P, pmin, e, d)
    return base

def total_cost(P, a, b, c, pmin, e=None, d=None, use_vpe=False):
    return float(np.sum(unit_cost(P, a, b, c, pmin, e, d, use_vpe)))

def emission_vector(P, alpha, beta, gamma, xi, delta):
    # E_i = 10^-2 (alpha + beta P + gamma P^2) + xi * exp(delta P)
    return 1e-2*(alpha + beta*P + gamma*(P**2)) + xi*np.exp(delta*P)

def total_emission(P, alpha, beta, gamma, xi, delta):
    return float(np.sum(emission_vector(P, alpha, beta, gamma, xi, delta)))

def compute_global_ppf(a,b,c,pmin,pmax, alpha,beta,gamma,xi,delta, e=None,d=None, use_vpe=False):
    c_min = total_cost(pmin, a,b,c,pmin, e,d, use_vpe)
    c_max = total_cost(pmax, a,b,c,pmin, e,d, use_vpe)
    e_min = total_emission(pmin, alpha,beta,gamma,xi,delta)
    e_max = total_emission(pmax, alpha,beta,gamma,xi,delta)
    denom = max(1e-12, (e_max - e_min))
    return (c_max - c_min) / denom

# ===================== DOA (unchanged structure; objective extended) =====================
def calculate_power(demand, a, b, c, pmin, pmax, rampup, rampdown,
                    mode='economic', ppf_h=None,
                    alpha=None, beta=None, gamma=None, xi=None, delta=None,
                    use_vpe=False, d=None, e=None,
                    pop=30, max_iter=100):
    dim = len(pmin)
    lb, ub = pmin, pmax

    if mode not in ('economic','emission','ceed'):
        mode = 'economic'
    if mode in ('emission','ceed') and ppf_h is None and \
       (alpha is not None and beta is not None and gamma is not None and xi is not None and delta is not None):
        ppf_h = compute_global_ppf(a,b,c,lb,ub,alpha,beta,gamma,xi,delta,e,d,use_vpe)

    # Objective: cost/emission/combined + penalty for demand mismatch
    def fobj(pos):
        cost_val = total_cost(pos, a,b,c, lb, e,d, use_vpe)
        if alpha is not None:
            emis_val = total_emission(pos, alpha,beta,gamma,xi,delta)
        else:
            emis_val = 0.0

        if mode == 'economic':
            obj = cost_val
        elif mode == 'emission':
            obj = emis_val
        else:  # 'ceed'
            h = ppf_h if ppf_h is not None else 1.0
            obj = cost_val + h * emis_val

        penalty = abs(demand - np.sum(pos)) * 10000.0
        return obj + penalty

    # Initialization
    x = np.random.uniform(lb, ub, (pop, dim))
    fbest = np.inf
    sbest = np.ones(dim)
    sbestd = np.tile(np.ones(dim), (5, 1))
    fbestd = np.ones(5) * np.inf
    fbest_history = np.ones(max_iter)
    SELECT = np.arange(pop)

    # ---- Exploration phase (first 90% iterations) ----
    for it in range(int(0.9 * max_iter)):
        for m in range(5):
            low = max(1, int(np.ceil(dim / (8 * (m + 1)))))
            high_incl = max(2, int(np.ceil(dim / (3 * (m + 1)))))
            k = low if low >= high_incl else np.random.randint(low, high_incl + 1)

            start = int((m) * pop / 5)
            end = int((m + 1) * pop / 5)

            # Update group best
            for j in range(start, end):
                fit = fobj(x[j])
                if fit < fbestd[m]:
                    fbestd[m] = fit
                    sbestd[m] = x[j].copy()

            # Memory & forgetting/supplementation
            for j in range(start, end):
                x[j] = sbestd[m].copy()
                indices = np.random.permutation(dim)[:k]
                if np.random.rand() < 0.9:
                    for h_idx in indices:
                        step = (np.random.rand() * (ub[h_idx] - lb[h_idx]) + lb[h_idx]) * \
                               (np.cos((it + max_iter / 10) * np.pi / max_iter) + 1) / 2
                        x[j, h_idx] += step
                        if x[j, h_idx] > ub[h_idx] or x[j, h_idx] < lb[h_idx]:
                            if dim > 15:
                                sel = np.random.choice(np.delete(SELECT, j))
                                x[j, h_idx] = x[sel, h_idx]
                            else:
                                x[j, h_idx] = np.random.rand() * (ub[h_idx] - lb[h_idx]) + lb[h_idx]
                else:
                    for h_idx in indices:
                        x[j, h_idx] = x[np.random.randint(pop), h_idx]

            # Update global best
            if fbestd[m] < fbest:
                fbest = fbestd[m]
                sbest = sbestd[m].copy()

        fbest_history[it] = fbest

    # ---- Exploitation phase (last 10% iterations) ----
    for it in range(int(0.9 * max_iter), max_iter):
        for p in range(pop):
            fit = fobj(x[p])
            if fit < fbest:
                fbest = fit
                sbest = x[p].copy()

        # Local exploitation around sbest
        for j in range(pop):
            km = max(2, int(np.ceil(dim / 3)))
            k = 2 if km == 2 else np.random.randint(2, km + 1)
            x[j] = sbest.copy()
            indices = np.random.permutation(dim)[:k]
            for h_idx in indices:
                step = (np.random.rand() * (ub[h_idx] - lb[h_idx]) + lb[h_idx]) * \
                       (np.cos(it * np.pi / max_iter) + 1) / 2
                x[j, h_idx] += step
                if x[j, h_idx] > ub[h_idx] or x[j, h_idx] < lb[h_idx]:
                    if dim > 15:
                        sel = np.random.choice(np.delete(SELECT, j))
                        x[j, h_idx] = x[sel, h_idx]
                    else:
                        x[j, h_idx] = np.random.rand() * (ub[h_idx] - lb[h_idx]) + lb[h_idx]

        fbest_history[it] = fbest

    # Scale final best solution to meet demand
    total_power = np.sum(sbest)
    if total_power > 0:
        sbest *= demand / total_power
    sbest = np.clip(sbest, lb, ub)

    # Ramp rate adjustment (relative to x[0] as in your original code)
    diff = sbest - x[0]
    up_mask, down_mask = diff > 0, diff < 0
    sbest[up_mask] = x[0][up_mask] + np.minimum(diff[up_mask], rampup[up_mask])
    sbest[down_mask] = x[0][down_mask] + np.maximum(diff[down_mask], -rampdown[down_mask])

    return sbest, max_iter

# ===================== RUN =====================
all_outputs = []
schedulling = []
summary_rows = []

# Precompute global penalty factor if needed
if DISPATCH_MODE in ('emission','ceed') and PPF_H is None:
    PPF_H = compute_global_ppf(a,b,c,Minimum_Capacity,Maximum_Capacity,
                               alpha,beta,gamma,xi,delta,
                               e=e_vpe, d=d_vpe, use_vpe=use_vpe)

for load_counter, demand in enumerate(power_demand, start=1):
    start_time = time.time()

    P, iterations = calculate_power(
        demand, a, b, c, Minimum_Capacity, Maximum_Capacity, ramp_up, ramp_down,
        mode=DISPATCH_MODE, ppf_h=PPF_H,
        alpha=alpha, beta=beta, gamma=gamma, xi=xi, delta=delta,
        use_vpe=use_vpe, d=d_vpe, e=e_vpe
    )

    # Costs & Emissions
    Fuel_Cost_i = unit_cost(P, a, b, c, Minimum_Capacity, e_vpe, d_vpe, use_vpe)
    Total_Fuel_Cost = float(np.sum(Fuel_Cost_i))
    Emission_i = emission_vector(P, alpha, beta, gamma, xi, delta)
    Total_Emission = float(np.sum(Emission_i))

    end_time = time.time()

    # Output Results (console)
    output = pd.DataFrame({
        'Unit': Unit.astype(int),
        'Power Produced (MW)': P,
        'Fuel Cost (Rp)': Fuel_Cost_i,
        'Emission (u)': Emission_i
    })
    print(f"Load Counter: {load_counter} | Demand: {demand:.2f} MW | Mode: {DISPATCH_MODE} (h={PPF_H:.6g})")
    print(output)
    print(f"Total Power Produced: {np.sum(P):.6f} MW")
    print(f"Total Fuel Cost: Rp {Total_Fuel_Cost}")
    print(f"Total Emission: {Total_Emission}")
    print(f'Computation Time: {end_time - start_time:.6f} s')
    print('------------------------------------------------\n')

    all_outputs.append({
        'Demand': demand,
        'Output': output,
        'Total Power Produced': float(np.sum(P)),
        'Total Fuel Cost': float(Total_Fuel_Cost),
        'Total Emission': float(Total_Emission),
        'Computation Time': float(end_time - start_time)
    })

    summary_rows.append({
        'Load Counter': load_counter,
        'Demand (MW)': float(demand),
        'Total Power Produced (MW)': float(np.sum(P)),
        'Total Fuel Cost (Rp)': float(Total_Fuel_Cost),
        'Total Emission (u)': float(Total_Emission),
        'PPF h': float(PPF_H if PPF_H is not None else 0.0),
        'Mode': DISPATCH_MODE,
        'Iterations': iterations,
        'Computation Time (s)': float(end_time - start_time),
    })

    # Scheduling (per-unit)
    info = {'Demand': demand}
    for idx, u in enumerate(Unit.astype(int)):
        info[f'Unit {u} Power (MW)'] = P[idx]
        info[f'Unit {u} Cost (Rp)'] = Fuel_Cost_i[idx]
        info[f'Unit {u} Emission (u)'] = Emission_i[idx]
    schedulling.append(info)

# ===================== POST: RAMP & LIMIT CHECKS =====================
ramp_rates, generation_limits = [], []
for i in range(1, len(all_outputs)):
    prev = all_outputs[i-1]['Output']['Power Produced (MW)'].values
    curr = all_outputs[i]['Output']['Power Produced (MW)'].values
    rr = curr - prev
    violations = np.logical_or(rr > ramp_up, rr < -ramp_down)
    ramp_rate_info = {'Demand': all_outputs[i]['Demand']}
    generation_limit_info = {'Demand': all_outputs[i]['Demand']}
    for j, u in enumerate(Unit.astype(int)):
        ramp_rate_info[f'Unit {u} ΔP (MW)'] = rr[j]
        ramp_rate_info[f'Unit {u} Violation'] = bool(violations[j])
        gen_violation = (curr[j] < Minimum_Capacity[j]) or (curr[j] > Maximum_Capacity[j])
        generation_limit_info[f'Unit {u} Output (MW)'] = curr[j]
        generation_limit_info[f'Unit {u} Violation'] = bool(gen_violation)
    ramp_rates.append(ramp_rate_info)
    generation_limits.append(generation_limit_info)

# ===================== SAVE TO EXCEL =====================
output_file_path = '/Users/workk/Documents/[Think!]/[FMK]/Research Things/Books/Optimasi/DOA (EED) 13u.xlsx'
with pd.ExcelWriter(output_file_path, engine='openpyxl') as writer:
    pd.DataFrame(schedulling).to_excel(writer, sheet_name='Schedulling', index=False)
    pd.DataFrame(summary_rows).to_excel(writer, sheet_name='Summary', index=False)
    pd.DataFrame(ramp_rates).to_excel(writer, sheet_name='Ramp Rates', index=False)
    pd.DataFrame(generation_limits).to_excel(writer, sheet_name='Generation Limits', index=False)

print("All results saved to 'DOA (EED) 13u.xlsx'")

Load Counter: 1 | Demand: 1800.00 MW | Mode: ceed (h=27.9728)
    Unit  Power Produced (MW)  Fuel Cost (Rp)  Emission (u)
0      1            35.548734      838.298588      1.498293
1      2           329.544365     3039.125070     32.187486
2      3           327.545086     3020.195235     31.723798
3      4           131.607757     1314.762790      2.004338
4      5           129.391576     1295.735460      1.849984
5      6           155.767916     1524.257873      3.989610
6      7           132.568385     1323.020283      2.072687
7      8           140.524471     1391.640097      2.672348
8      9           121.317184     1226.680872      1.326772
9     10            51.932507      580.278995     -0.024334
10    11            85.321383      880.438353      1.077706
11    12            78.557621      819.122035      2.252750
12    13            80.543914      837.101651      2.366055
Total Power Produced: 1800.170900 MW
Total Fuel Cost: Rp 18090.657301418327
Total Emission: 84.997

In [3]:
import pandas as pd
import numpy as np
import time
import itertools

# ===================== SETTINGS =====================
DISPATCH_MODE = 'ceed'      # 'economic' | 'emission' | 'ceed'
PPF_H = None                # None -> auto compute global price-penalty; or set a float, e.g., 1.0
USE_VPE_IF_AVAILABLE = False  # True to include Valve-Point Effect if 'd' and 'e' columns exist

# ===================== LOAD DATA =====================
data = pd.read_excel(
    '/Users/workk/Documents/[Think!]/[FMK]/Research Things/Books/Optimasi/Dataset/40-unit.xlsx',
    sheet_name='Table 3'
)

def get_col(df, names, default=0.0):
    for n in names:
        if n in df.columns: 
            return df[n].values.astype(float)
    return np.full(len(df), default, dtype=float)

a = get_col(data, ['a'])
b = get_col(data, ['b'])
c = get_col(data, ['c'])
Minimum_Capacity = get_col(data, ['pmin','Pmin','Pmin (MW)'])
Maximum_Capacity = get_col(data, ['pmax','Pmax','Pmax (MW)'])
ramp_up        = get_col(data, ['rampup','RU','RampUp'])
ramp_down      = get_col(data, ['rampdown','RD','RampDown'])
Unit = get_col(data, ['Unit','unit','Unit ke-']).astype(int)

# Optional: VPE coefficients
d_vpe = get_col(data, ['d','D'], default=0.0)
e_vpe = get_col(data, ['e','E'], default=0.0)
use_vpe = USE_VPE_IF_AVAILABLE and (('d' in data.columns or 'D' in data.columns) and ('e' in data.columns or 'E' in data.columns))

# Emission coefficients (handle latin or Greek headers)
alpha = get_col(data, ['alpha','Alpha','α'])
beta  = get_col(data, ['beta','Beta','β'])
gamma = get_col(data, ['gamma','Gamma','γ'])
xi    = get_col(data, ['xi','Xi','ξ'])
delta = get_col(data, ['delta','Delta','δ'])

# Demand
power_demand_data = pd.read_excel('/Users/workk/Documents/[Think!]/[FMK]/Research Things/Books/Optimasi/Dataset/pd40u.xlsx')
power_demand = power_demand_data['load'].values.astype(float)

# ===================== COST & EMISSION =====================
def vpe_term(P, pmin, e, d):
    # | e_i * sin( d_i * (pmin_i - P_i) ) |
    return np.abs(e * np.sin(d * (pmin - P)))

def unit_cost(P, a, b, c, pmin, e=None, d=None, use_vpe=False):
    base = a + b*P + c*(P**2)
    if use_vpe and e is not None and d is not None:
        return base + vpe_term(P, pmin, e, d)
    return base

def total_cost(P, a, b, c, pmin, e=None, d=None, use_vpe=False):
    return float(np.sum(unit_cost(P, a, b, c, pmin, e, d, use_vpe)))

def emission_vector(P, alpha, beta, gamma, xi, delta):
    # E_i = 10^-2 (alpha + beta P + gamma P^2) + xi * exp(delta P)
    return 1e-2*(alpha + beta*P + gamma*(P**2)) + xi*np.exp(delta*P)

def total_emission(P, alpha, beta, gamma, xi, delta):
    return float(np.sum(emission_vector(P, alpha, beta, gamma, xi, delta)))

def compute_global_ppf(a,b,c,pmin,pmax, alpha,beta,gamma,xi,delta, e=None,d=None, use_vpe=False):
    c_min = total_cost(pmin, a,b,c,pmin, e,d, use_vpe)
    c_max = total_cost(pmax, a,b,c,pmin, e,d, use_vpe)
    e_min = total_emission(pmin, alpha,beta,gamma,xi,delta)
    e_max = total_emission(pmax, alpha,beta,gamma,xi,delta)
    denom = max(1e-12, (e_max - e_min))
    return (c_max - c_min) / denom

# ===================== DOA (unchanged structure; objective extended) =====================
def calculate_power(demand, a, b, c, pmin, pmax, rampup, rampdown,
                    mode='economic', ppf_h=None,
                    alpha=None, beta=None, gamma=None, xi=None, delta=None,
                    use_vpe=False, d=None, e=None,
                    pop=30, max_iter=100):
    dim = len(pmin)
    lb, ub = pmin, pmax

    if mode not in ('economic','emission','ceed'):
        mode = 'economic'
    if mode in ('emission','ceed') and ppf_h is None and \
       (alpha is not None and beta is not None and gamma is not None and xi is not None and delta is not None):
        ppf_h = compute_global_ppf(a,b,c,lb,ub,alpha,beta,gamma,xi,delta,e,d,use_vpe)

    # Objective: cost/emission/combined + penalty for demand mismatch
    def fobj(pos):
        cost_val = total_cost(pos, a,b,c, lb, e,d, use_vpe)
        if alpha is not None:
            emis_val = total_emission(pos, alpha,beta,gamma,xi,delta)
        else:
            emis_val = 0.0

        if mode == 'economic':
            obj = cost_val
        elif mode == 'emission':
            obj = emis_val
        else:  # 'ceed'
            h = ppf_h if ppf_h is not None else 1.0
            obj = cost_val + h * emis_val

        penalty = abs(demand - np.sum(pos)) * 10000.0
        return obj + penalty

    # Initialization
    x = np.random.uniform(lb, ub, (pop, dim))
    fbest = np.inf
    sbest = np.ones(dim)
    sbestd = np.tile(np.ones(dim), (5, 1))
    fbestd = np.ones(5) * np.inf
    fbest_history = np.ones(max_iter)
    SELECT = np.arange(pop)

    # ---- Exploration phase (first 90% iterations) ----
    for it in range(int(0.9 * max_iter)):
        for m in range(5):
            low = max(1, int(np.ceil(dim / (8 * (m + 1)))))
            high_incl = max(2, int(np.ceil(dim / (3 * (m + 1)))))
            k = low if low >= high_incl else np.random.randint(low, high_incl + 1)

            start = int((m) * pop / 5)
            end = int((m + 1) * pop / 5)

            # Update group best
            for j in range(start, end):
                fit = fobj(x[j])
                if fit < fbestd[m]:
                    fbestd[m] = fit
                    sbestd[m] = x[j].copy()

            # Memory & forgetting/supplementation
            for j in range(start, end):
                x[j] = sbestd[m].copy()
                indices = np.random.permutation(dim)[:k]
                if np.random.rand() < 0.9:
                    for h_idx in indices:
                        step = (np.random.rand() * (ub[h_idx] - lb[h_idx]) + lb[h_idx]) * \
                               (np.cos((it + max_iter / 10) * np.pi / max_iter) + 1) / 2
                        x[j, h_idx] += step
                        if x[j, h_idx] > ub[h_idx] or x[j, h_idx] < lb[h_idx]:
                            if dim > 15:
                                sel = np.random.choice(np.delete(SELECT, j))
                                x[j, h_idx] = x[sel, h_idx]
                            else:
                                x[j, h_idx] = np.random.rand() * (ub[h_idx] - lb[h_idx]) + lb[h_idx]
                else:
                    for h_idx in indices:
                        x[j, h_idx] = x[np.random.randint(pop), h_idx]

            # Update global best
            if fbestd[m] < fbest:
                fbest = fbestd[m]
                sbest = sbestd[m].copy()

        fbest_history[it] = fbest

    # ---- Exploitation phase (last 10% iterations) ----
    for it in range(int(0.9 * max_iter), max_iter):
        for p in range(pop):
            fit = fobj(x[p])
            if fit < fbest:
                fbest = fit
                sbest = x[p].copy()

        # Local exploitation around sbest
        for j in range(pop):
            km = max(2, int(np.ceil(dim / 3)))
            k = 2 if km == 2 else np.random.randint(2, km + 1)
            x[j] = sbest.copy()
            indices = np.random.permutation(dim)[:k]
            for h_idx in indices:
                step = (np.random.rand() * (ub[h_idx] - lb[h_idx]) + lb[h_idx]) * \
                       (np.cos(it * np.pi / max_iter) + 1) / 2
                x[j, h_idx] += step
                if x[j, h_idx] > ub[h_idx] or x[j, h_idx] < lb[h_idx]:
                    if dim > 15:
                        sel = np.random.choice(np.delete(SELECT, j))
                        x[j, h_idx] = x[sel, h_idx]
                    else:
                        x[j, h_idx] = np.random.rand() * (ub[h_idx] - lb[h_idx]) + lb[h_idx]

        fbest_history[it] = fbest

    # Scale final best solution to meet demand
    total_power = np.sum(sbest)
    if total_power > 0:
        sbest *= demand / total_power
    sbest = np.clip(sbest, lb, ub)

    # Ramp rate adjustment (relative to x[0] as in your original code)
    diff = sbest - x[0]
    up_mask, down_mask = diff > 0, diff < 0
    sbest[up_mask] = x[0][up_mask] + np.minimum(diff[up_mask], rampup[up_mask])
    sbest[down_mask] = x[0][down_mask] + np.maximum(diff[down_mask], -rampdown[down_mask])

    return sbest, max_iter

# ===================== RUN =====================
all_outputs = []
schedulling = []
summary_rows = []

# Precompute global penalty factor if needed
if DISPATCH_MODE in ('emission','ceed') and PPF_H is None:
    PPF_H = compute_global_ppf(a,b,c,Minimum_Capacity,Maximum_Capacity,
                               alpha,beta,gamma,xi,delta,
                               e=e_vpe, d=d_vpe, use_vpe=use_vpe)

for load_counter, demand in enumerate(power_demand, start=1):
    start_time = time.time()

    P, iterations = calculate_power(
        demand, a, b, c, Minimum_Capacity, Maximum_Capacity, ramp_up, ramp_down,
        mode=DISPATCH_MODE, ppf_h=PPF_H,
        alpha=alpha, beta=beta, gamma=gamma, xi=xi, delta=delta,
        use_vpe=use_vpe, d=d_vpe, e=e_vpe
    )

    # Costs & Emissions
    Fuel_Cost_i = unit_cost(P, a, b, c, Minimum_Capacity, e_vpe, d_vpe, use_vpe)
    Total_Fuel_Cost = float(np.sum(Fuel_Cost_i))
    Emission_i = emission_vector(P, alpha, beta, gamma, xi, delta)
    Total_Emission = float(np.sum(Emission_i))

    end_time = time.time()

    # Output Results (console)
    output = pd.DataFrame({
        'Unit': Unit.astype(int),
        'Power Produced (MW)': P,
        'Fuel Cost (Rp)': Fuel_Cost_i,
        'Emission (u)': Emission_i
    })
    print(f"Load Counter: {load_counter} | Demand: {demand:.2f} MW | Mode: {DISPATCH_MODE} (h={PPF_H:.6g})")
    print(output)
    print(f"Total Power Produced: {np.sum(P):.6f} MW")
    print(f"Total Fuel Cost: Rp {Total_Fuel_Cost}")
    print(f"Total Emission: {Total_Emission}")
    print(f'Computation Time: {end_time - start_time:.6f} s')
    print('------------------------------------------------\n')

    all_outputs.append({
        'Demand': demand,
        'Output': output,
        'Total Power Produced': float(np.sum(P)),
        'Total Fuel Cost': float(Total_Fuel_Cost),
        'Total Emission': float(Total_Emission),
        'Computation Time': float(end_time - start_time)
    })

    summary_rows.append({
        'Load Counter': load_counter,
        'Demand (MW)': float(demand),
        'Total Power Produced (MW)': float(np.sum(P)),
        'Total Fuel Cost (Rp)': float(Total_Fuel_Cost),
        'Total Emission (u)': float(Total_Emission),
        'PPF h': float(PPF_H if PPF_H is not None else 0.0),
        'Mode': DISPATCH_MODE,
        'Iterations': iterations,
        'Computation Time (s)': float(end_time - start_time),
    })

    # Scheduling (per-unit)
    info = {'Demand': demand}
    for idx, u in enumerate(Unit.astype(int)):
        info[f'Unit {u} Power (MW)'] = P[idx]
        info[f'Unit {u} Cost (Rp)'] = Fuel_Cost_i[idx]
        info[f'Unit {u} Emission (u)'] = Emission_i[idx]
    schedulling.append(info)

# ===================== POST: RAMP & LIMIT CHECKS =====================
ramp_rates, generation_limits = [], []
for i in range(1, len(all_outputs)):
    prev = all_outputs[i-1]['Output']['Power Produced (MW)'].values
    curr = all_outputs[i]['Output']['Power Produced (MW)'].values
    rr = curr - prev
    violations = np.logical_or(rr > ramp_up, rr < -ramp_down)
    ramp_rate_info = {'Demand': all_outputs[i]['Demand']}
    generation_limit_info = {'Demand': all_outputs[i]['Demand']}
    for j, u in enumerate(Unit.astype(int)):
        ramp_rate_info[f'Unit {u} ΔP (MW)'] = rr[j]
        ramp_rate_info[f'Unit {u} Violation'] = bool(violations[j])
        gen_violation = (curr[j] < Minimum_Capacity[j]) or (curr[j] > Maximum_Capacity[j])
        generation_limit_info[f'Unit {u} Output (MW)'] = curr[j]
        generation_limit_info[f'Unit {u} Violation'] = bool(gen_violation)
    ramp_rates.append(ramp_rate_info)
    generation_limits.append(generation_limit_info)

# ===================== SAVE TO EXCEL =====================
output_file_path = '/Users/workk/Documents/[Think!]/[FMK]/Research Things/Books/Optimasi/DOA (EED) 40u.xlsx'
with pd.ExcelWriter(output_file_path, engine='openpyxl') as writer:
    pd.DataFrame(schedulling).to_excel(writer, sheet_name='Schedulling', index=False)
    pd.DataFrame(summary_rows).to_excel(writer, sheet_name='Summary', index=False)
    pd.DataFrame(ramp_rates).to_excel(writer, sheet_name='Ramp Rates', index=False)
    pd.DataFrame(generation_limits).to_excel(writer, sheet_name='Generation Limits', index=False)

print("All results saved to 'DOA (EED) 40u.xlsx'")

Load Counter: 1 | Demand: 10500.00 MW | Mode: ceed (h=0.000173776)
    Unit  Power Produced (MW)  Fuel Cost (Rp)  Emission (u)
0      1            55.392177      488.665576  3.257344e+01
1      2            47.248593      428.091795  2.069483e+01
2      3           114.199780     1381.415884  6.650176e+06
3      4           118.126978     1466.755196  6.610864e+01
4      5            57.085870      491.449687  1.139836e+01
5      6           116.484601     1314.985158  2.266387e+02
6      7           261.545043     2632.125438  1.128706e+03
7      8           299.636196     2928.163712  4.678989e+03
8      9           281.913691     2771.783993  2.022089e+03
9     10           151.051616     2809.426222  2.748793e+03
10    11           314.026114     5193.990733  5.004699e+03
11    12           301.786084     5035.767715  3.537745e+03
12    13           389.992093     6428.616194  1.677818e+03
13    14           482.019668     7768.672929  1.121737e+04
14    15           357.766515    